> LangChain Level 1：朴素基础检索

- 这份 notebook 用 8 个工程进度 TXT 演示**通用检索基线**：

- **TXT → Document → chunk → embedding → InMemoryVectorStore → Dense / BM25 → Weighted RRF**

- 关注“怎样召回候选”，不解析工程别名、父子任务或日期字段，也不把相似 chunk直接当成最终答案。工程进度专用的路由、完整任务记录和 Evidence Gate 放在static_retrieve_langchain_level1a.ipynb。

> 1. 导入与本地路径

当前小知识库使用 InMemoryVectorStore 足够：向量可以只存在当前 Python 进程中，不涉及持久化或 ANN 索引。

In [1]:
from __future__ import annotations

import hashlib
import re
import unicodedata
from collections import defaultdict
from pathlib import Path

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi


repo_root = Path.cwd().resolve()
if repo_root.name == "ZZworkbench":
    repo_root = repo_root.parent

text_dir = repo_root / "knowledge" / "project_progress" / "texts" / "v4"
embed_model_name = "iic--nlp_gte_sentence-embedding_chinese-base"
embed_path = Path("/mnt/e/local_models/embedding") / embed_model_name

assert text_dir.is_dir(), f"找不到语料目录：{text_dir}"
assert embed_path.is_dir(), "找不到本地 embedding 模型目录"
print({"repo_ok": repo_root.name == "pipelines_rag", "model": embed_path.name})

{'repo_ok': True, 'model': 'iic--nlp_gte_sentence-embedding_chinese-base'}


> 2. TXT → Document

使用 sorted 固定文件顺序。metadata 只包含通用字段；source 保存仓库相对路径，
避免输出用户主目录。document_id 由相对路径和内容生成，语料不变时 ID 就不变。

In [2]:
documents: list[Document] = []
for path in sorted(text_dir.glob("*.txt")):
    text = path.read_text(encoding="utf-8")
    relative_source = path.relative_to(repo_root).as_posix()
    document_id = "doc-" + hashlib.sha1(
        f"{relative_source}\n{text}".encode("utf-8")
    ).hexdigest()[:12]
    documents.append(
        Document(
            id=document_id,
            page_content=text,
            metadata={
                "source": relative_source,
                "source_name": path.name,
                "version": path.parent.name,
                "document_id": document_id,
            },
        )
    )

print({"documents": len(documents), "sources": [d.metadata["source_name"] for d in documents]})
assert len(documents) == 8

{'documents': 8, 'sources': ['110kV黄金输变电工程三级进度计划.txt', '110千伏节点计划-重点关注.txt', '三级进度计划-土建.txt', '三虎输变电工程三级进度计划土建部分.txt', '三虎输变电工程三级进度计划电气部分.txt', '南溪三级进度.txt', '珠海110kV江湾输变电工程总体进度计划横道图.txt', '珠海110千伏江湾输变电工程施工进度计划（202.txt']}


> 3. Document → chunk

RecursiveCharacterTextSplitter 只是普通字符切分器。中文标点放在 separators 前部，
add_start_index 保留 chunk 在父文档中的起始位置(这个无法确定是否有用)。

chunk_size 和 overlap 是本实验参数，不是通用最佳值。此处只建立基础检索单位，
不保证每个 chunk 都对应一条完整工程任务记录。

In [3]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", "。", "！", "？", "；", "，", " ", ""],
    chunk_size=640,
    chunk_overlap=128,
    add_start_index=True,
)
raw_chunks = splitter.split_documents(documents)

chunks: list[Document] = []
for index, chunk in enumerate(raw_chunks):
    chunk_id = f"{chunk.metadata['document_id']}:chunk-{index:03d}"
    chunks.append(
        Document(
            id=chunk_id,
            page_content=chunk.page_content,
            metadata={**chunk.metadata, "chunk_id": chunk_id},
        )
    )

lengths = [len(chunk.page_content) for chunk in chunks]
print(
    {
        "chunks": len(chunks),
        "min_chars": min(lengths),
        "max_chars": max(lengths),
        "mean_chars": round(sum(lengths) / len(lengths), 1),
    }
)
# lightrag 有提到， R-splitter 也有可能得到一长串文本，这个时候需要 打补丁 处理一下。


{'chunks': 88, 'min_chars': 127, 'max_chars': 637, 'mean_chars': 575.1}


> 4. embedding → InMemoryVectorStore

HuggingFaceEmbeddings 使用本地中文模型，并归一化 document/query 向量。
InMemoryVectorStore 当前实现使用 cosine similarity 做精确线性扫描；数据量只有几十个
chunk，不需要持久化向量数据库。

In [4]:
embed_model = HuggingFaceEmbeddings(
    model=str(embed_path),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
    query_encode_kwargs={"normalize_embeddings": True},
    show_progress=False,
)

vector_store = InMemoryVectorStore(embedding=embed_model)
chunk_ids = [str(chunk.id) for chunk in chunks]
indexed_ids = vector_store.add_documents(documents=chunks, ids=chunk_ids)

print({"store": type(vector_store).__name__, "indexed_chunks": len(indexed_ids)})
assert len(indexed_ids) == len(chunks)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'store': 'InMemoryVectorStore', 'indexed_chunks': 88}


> 5. VectorStore 基础接口

- similarity_search：返回 Document；
- similarity_search_with_score：返回 (Document, score)，便于调试；
- similarity_search_by_vector：查询向量已经计算好时使用；
- get_by_ids：按稳定 ID 回读。

对当前 InMemoryVectorStore，with_score 是 cosine similarity，越大越相似。
其他 VectorStore 可能返回距离，使用前必须查看具体实现。

In [5]:
query = "珠海黄金输变电工程的主体结构封顶计划什么时候完成？"

plain_docs = vector_store.similarity_search(query, k=3)
scored_docs = vector_store.similarity_search_with_score(query, k=3)
query_vector = embed_model.embed_query(query)
vector_docs = vector_store.similarity_search_by_vector(query_vector, k=2)
fetched_docs = vector_store.get_by_ids(chunk_ids[:2])

print("similarity_search:", [doc.metadata["chunk_id"] for doc in plain_docs])
print(
    "with_score:",
    [
        {
            "chunk_id": doc.metadata["chunk_id"],
            "cosine": round(float(score), 4),
        }
        for doc, score in scored_docs
    ],
)
print({"by_vector": len(vector_docs), "get_by_ids": len(fetched_docs)})

similarity_search: ['doc-a7a3bb55d834:chunk-000', 'doc-59c80049d501:chunk-047', 'doc-7f664f15e764:chunk-073']
with_score: [{'chunk_id': 'doc-a7a3bb55d834:chunk-000', 'cosine': 0.8058}, {'chunk_id': 'doc-59c80049d501:chunk-047', 'cosine': 0.7942}, {'chunk_id': 'doc-7f664f15e764:chunk-073', 'cosine': 0.7894}]
{'by_vector': 2, 'get_by_ids': 2}


> 6. as_retriever：similarity 与 MMR

as_retriever 把 VectorStore 包装成 LangChain Runnable，可统一调用 invoke/batch。

- similarity：k 是最终返回数量，filter 是 metadata 过滤函数；
- MMR（Maximal Marginal Relevance）：先从 fetch_k 个 Dense 候选中选择 k 个，
  lambda_mult 越接近 1 越偏相关，越接近 0 越偏多样；
- MMR 不是新的 embedding 或 ANN 算法，也不保证单条事实查询更准。

In [6]:
source_name = "110kV黄金输变电工程三级进度计划.txt"
source_filter = lambda doc: doc.metadata.get("source_name") == source_name

similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4, "filter": source_filter},
)
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 16,
        "lambda_mult": 0.65,
        "filter": source_filter,
    },
)

retriever_results = {
    "similarity": similarity_retriever.invoke(query),
    "mmr": mmr_retriever.invoke(query),
}
print(
    {
        name: [doc.metadata["chunk_id"] for doc in docs]
        for name, docs in retriever_results.items()
    }
)

{'similarity': ['doc-a7a3bb55d834:chunk-000', 'doc-a7a3bb55d834:chunk-005', 'doc-a7a3bb55d834:chunk-002', 'doc-a7a3bb55d834:chunk-001'], 'mmr': ['doc-a7a3bb55d834:chunk-000', 'doc-a7a3bb55d834:chunk-005', 'doc-a7a3bb55d834:chunk-002', 'doc-a7a3bb55d834:chunk-001']}


> 7. 手动 similarity threshold

similarity_score_threshold 能避免 Top-K 强制返回无关结果，但当前安装版
InMemoryVectorStore 没有实现 relevance-score 映射，直接使用该 retriever 会抛
NotImplementedError。

这里用 with_score 返回的原始 cosine similarity 手动过滤。0.55 只用于展示接口，
不能当作通用阈值；真实阈值要用当前模型、当前语料和标注集校准。

In [7]:
threshold = 0.55
threshold_candidates = [
    (doc, float(score))
    for doc, score in vector_store.similarity_search_with_score(query, k=8)
    if float(score) >= threshold
]
print(
    [
        {
            "chunk_id": doc.metadata["chunk_id"],
            "score": round(score, 4),
        }
        for doc, score in threshold_candidates
    ]
)

[{'chunk_id': 'doc-a7a3bb55d834:chunk-000', 'score': 0.8058}, {'chunk_id': 'doc-59c80049d501:chunk-047', 'score': 0.7942}, {'chunk_id': 'doc-7f664f15e764:chunk-073', 'score': 0.7894}, {'chunk_id': 'doc-85f2d371674d:chunk-050', 'score': 0.7863}, {'chunk_id': 'doc-40446a836767:chunk-053', 'score': 0.7848}, {'chunk_id': 'doc-d2a1e7feb267:chunk-067', 'score': 0.7813}, {'chunk_id': 'doc-280a6366e60c:chunk-063', 'score': 0.7754}, {'chunk_id': 'doc-59c80049d501:chunk-049', 'score': 0.76}]


> 8. 中文 BM25

BM25 接收 token 序列。中文没有天然空格，直接 split() 往往把整句当成一个 token。
这里采用简单、通用的混合 tokenizer：

- 中文连续文本生成重叠 2-gram / 3-gram；
- 英文、数字、日期和 110kV 等词面保留为归一化 token。

它不是完整中文分词器，但比直接 split 更适合当前包含工程名、任务名和编号的文本。

In [6]:
lexical_pattern = re.compile(
    r"[\u3400-\u9fff]+|[a-z0-9]+(?:[._/-][a-z0-9]+)*"
)


def tokenize(text: str) -> list[str]:
    normalized = unicodedata.normalize("NFKC", text).casefold()
    tokens: list[str] = []
    for match in lexical_pattern.finditer(normalized):
        value = match.group(0)
        if "\u3400" <= value[0] <= "\u9fff":
            if len(value) == 1:
                tokens.append(value)
            else:
                tokens.extend(value[i : i + 2] for i in range(len(value) - 1))
                tokens.extend(value[i : i + 3] for i in range(len(value) - 2))
        else:
            tokens.append(value)
    return tokens


tokenized_chunks = [
    tokenize(f"{chunk.metadata['source_name']}\n{chunk.page_content}")
    for chunk in chunks
]
bm25 = BM25Okapi(tokenized_chunks)


def bm25_search(query: str, k: int = 8) -> list[tuple[Document, float]]:
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(
        enumerate(scores),
        key=lambda item: (-float(item[1]), str(chunks[item[0]].id)),
    )
    return [
        (chunks[index], float(score))
        for index, score in ranked[:k]
        if float(score) > 0
    ]


print("str.split():", query.casefold().split())
print("tokenizer sample:", tokenize(query)[:24])
print(
    "BM25 top:",
    [
        (doc.metadata["chunk_id"], round(score, 3))
        for doc, score in bm25_search(query, k=3)
    ],
)

str.split(): ['珠海黄金输变电工程的主体结构封顶计划什么时候完成？']
tokenizer sample: ['珠海', '海黄', '黄金', '金输', '输变', '变电', '电工', '工程', '程的', '的主', '主体', '体结', '结构', '构封', '封顶', '顶计', '计划', '划什', '什么', '么时', '时候', '候完', '完成', '珠海黄']
BM25 top: [('doc-a7a3bb55d834:chunk-001', 38.73), ('doc-46dac70d3d16:chunk-017', 23.894), ('doc-a7a3bb55d834:chunk-000', 22.157)]


> 9. Weighted RRF

BM25 score 与 cosine similarity 不在同一尺度，不能直接相加。
Reciprocal Rank Fusion（RRF）只融合排名：

**score(d) = Σ weight(branch) / (rrf_k + rank(branch, d))**

权重只是当前实验起点。level1 不做工程 route，因此这里给 Dense 与 BM25 相同权重。

In [7]:
def weighted_rrf(
    rankings: dict[str, list[tuple[Document, float]]],
    weights: dict[str, float],
    k: int = 5,
    rrf_k: int = 60,
) -> list[dict]:
    scores: dict[str, float] = defaultdict(float)
    docs: dict[str, Document] = {}
    branch_ranks: dict[str, dict[str, int]] = defaultdict(dict)

    for branch, results in rankings.items():
        for rank, (doc, _raw_score) in enumerate(results, start=1):
            chunk_id = str(doc.id)
            docs[chunk_id] = doc
            scores[chunk_id] += weights[branch] / (rrf_k + rank)
            branch_ranks[chunk_id][branch] = rank

    ordered = sorted(scores, key=lambda key: (-scores[key], key))[:k]
    return [
        {
            "document": docs[chunk_id],
            "rrf_score": scores[chunk_id],
            "branch_ranks": branch_ranks[chunk_id],
        }
        for chunk_id in ordered
    ]


dense_ranking = vector_store.similarity_search_with_score(query, k=12)
lexical_ranking = bm25_search(query, k=12)
fused = weighted_rrf(
    {"dense": dense_ranking, "bm25": lexical_ranking},
    {"dense": 0.5, "bm25": 0.5},
    k=5,
)
print(
    [
        {
            "chunk_id": row["document"].metadata["chunk_id"],
            "source": row["document"].metadata["source_name"],
            "rrf": round(row["rrf_score"], 6),
            "ranks": row["branch_ranks"],
        }
        for row in fused
    ]
)

[{'chunk_id': 'doc-a7a3bb55d834:chunk-000', 'source': '110kV黄金输变电工程三级进度计划.txt', 'rrf': 0.016133, 'ranks': {'dense': 1, 'bm25': 3}}, {'chunk_id': 'doc-a7a3bb55d834:chunk-001', 'source': '110kV黄金输变电工程三级进度计划.txt', 'rrf': 0.008197, 'ranks': {'bm25': 1}}, {'chunk_id': 'doc-46dac70d3d16:chunk-017', 'source': '110千伏节点计划-重点关注.txt', 'rrf': 0.008065, 'ranks': {'bm25': 2}}, {'chunk_id': 'doc-59c80049d501:chunk-047', 'source': '三级进度计划-土建.txt', 'rrf': 0.008065, 'ranks': {'dense': 2}}, {'chunk_id': 'doc-7f664f15e764:chunk-073', 'source': '珠海110千伏江湾输变电工程施工进度计划（202.txt', 'rrf': 0.007937, 'ranks': {'dense': 3}}]


> 10. 三个真实问题的候选对照

这里只检查每条路线返回了什么候选，不解析最终日期。结果表保留来源、chunk_id 和排名，
便于观察 Dense、BM25 与融合结果的差异。

In [10]:
queries = [
    "珠海黄金输变电工程的主体结构封顶计划什么时候完成？",
    "南溪旅游输变电工程的地基基础施工计划起止时间是什么？",
    "施工准备什么时候完成？",
]

for current_query in queries:
    dense = vector_store.similarity_search_with_score(current_query, k=8)
    lexical = bm25_search(current_query, k=8)
    fusion = weighted_rrf(
        {"dense": dense, "bm25": lexical},
        {"dense": 0.5, "bm25": 0.5},
        k=3,
    )
    print("\nQUERY:", current_query)
    for rank, row in enumerate(fusion, start=1):
        doc = row["document"]
        preview = " ".join(doc.page_content.split())[:90]
        print(
            rank,
            doc.metadata["source_name"],
            doc.metadata["chunk_id"],
            row["branch_ranks"],
            preview,
        )


QUERY: 珠海黄金输变电工程的主体结构封顶计划什么时候完成？
1 110kV黄金输变电工程三级进度计划.txt doc-a7a3bb55d834:chunk-000 {'dense': 1, 'bm25': 3} 以下内容来自PDF第1页。 该进度计划的完整名称为珠海110千伏黄金输变电工程工程进度计划横道图。 标识号1是顶层独立任务“施工合同签定”。计划开始2024年4月25日，计划完成2
2 110kV黄金输变电工程三级进度计划.txt doc-a7a3bb55d834:chunk-001 {'bm25': 1} 标识号9是父任务“变电站土建”（标识号5）的第4个子任务“结构出0米”。计划开始2024年8月9日，计划完成2024年8月29日。 标识号10是父任务“变电站土建”（标识号5）的第
3 110千伏节点计划-重点关注.txt doc-46dac70d3d16:chunk-017 {'bm25': 2} 标识号64是父任务“四层主体结构（主控室层）”（标识号60）的第4个子任务“砼浇筑”。工期1 d。计划开始2028年11月10日，计划完成2028年11月10日。 标识号65是父任

QUERY: 南溪旅游输变电工程的地基基础施工计划起止时间是什么？
1 南溪三级进度.txt doc-280a6366e60c:chunk-063 {'dense': 1, 'bm25': 1} 以下内容来自PDF第1页。 该进度计划的完整名称为珠海110千伏南溪（旅游）输变电工程施工进度计划横道图。 标识号1是顶层独立任务“地基基础施工”。工期82工作日。计划开始2025
2 110kV黄金输变电工程三级进度计划.txt doc-a7a3bb55d834:chunk-000 {'dense': 5, 'bm25': 8} 以下内容来自PDF第1页。 该进度计划的完整名称为珠海110千伏黄金输变电工程工程进度计划横道图。 标识号1是顶层独立任务“施工合同签定”。计划开始2024年4月25日，计划完成2
3 三级进度计划-土建.txt doc-59c80049d501:chunk-047 {'dense': 2} 以下内容来自PDF第1页。 该进度计划的完整名称为珠海110千伏禾益输变电工程-变电站土建部分。 标识号1是顶层独立任务“施工准备”。工期10工作日。计划开始2


QUERY: 施工准备什么时候完成？
1 三级进度计划-土建.txt doc-59c80049d501:chunk-047 {'dense': 8, 'bm25': 1} 以下内容来自PDF第1页。 该进度计划的完整名称为珠海110千伏禾益输变电工程-变电站土建部分。 标识号1是顶层独立任务“施工准备”。工期10工作日。计划开始2024年8月20日，
2 三虎输变电工程三级进度计划土建部分.txt doc-85f2d371674d:chunk-052 {'dense': 1} 标识号16是顶层独立任务“8.500m层砌筑工程”。工期20个工作日。计划开始2022年12月28日，计划完成2023年1月23日。 标识号17是顶层独立任务“抹灰工程”。工期45
3 三虎输变电工程三级进度计划土建部分.txt doc-85f2d371674d:chunk-050 {'bm25': 2} 以下内容来自PDF第1页。 该进度计划的完整名称为珠海110千伏三虎输变电工程（不含通信部分）施工进度计划横道图-变电站土建部分。 标识号1是顶层独立任务“施工准备”。工期10个工


> 11. 结论

- InMemoryVectorStore 足够支持当前小型知识库的基础实验；
- Dense 处理语义近似，BM25 保留词面信号，RRF 避免混加不可比分数；
- Top-K 只是候选集合，不代表其中的日期一定属于用户询问的任务；
- MMR 解决冗余，threshold 解决最低相似度，两者都不等于事实校验；
- 下一份 level1a 才会针对工程进度数据加入工程路由、完整任务记录和引用/拒答。

> 12 补充一下 `VectorStore` 除了`存储`功能之外的，`索引`功能



```html
                    Vector Store
                         │
          ┌──────────────┴──────────────┐
          │                             │
      数据怎么保存？                 数据怎么搜索？
          │                             │
      Persistence                     Index
          │                             │
    内存 / 磁盘 / DB             Flat / HNSW / IVF ...

```



> ANN：不再把 query 和所有向量比较，而是通过提前构建的数据结构，快速找到“很可能是最近邻”的那部分向量。



| ANN 方法 | 会在哪看到 | 核心思想 |
|---|---|---|
| **HNSW** | Chroma、Qdrant、Weaviate、Milvus、pgvector 等 | 构造多层近邻图 |
| **IVF** | FAISS、Milvus | 把向量先聚类，再搜索相关簇 |
| **PQ** | FAISS、Milvus | 向量压缩，减少内存和计算 |
| **IVF-PQ** | FAISS、Milvus | 聚类 + 压缩 |
| Flat | FAISS 等 | 其实不是 ANN，而是暴力精确搜索 |



> ANN 索引，就是为了避免向量检索时逐个扫描全部 embedding，而预先构建一种向量搜索数据结构，用少量召回精度换取大规模近邻搜索速度。

> LangChain InMemoryVectorStore = 内存存储 + 全量余弦相似度计算 + Top-K 排序，没有专门的 ANN 索引。



```html
InMemoryVectorStore
│
├── 存储
│   └── Python 内存
│
└── 检索
    └── Flat / brute-force
        所有 vector 计算 cosine similarity
```


```html
Qdrant
│
├── 存储
│   └── 磁盘持久化
│
└── 检索
    └── HNSW ANN index
```

> Flat → HNSW → IVF → PQ 不是严格的“升级顺序”。
它们分别解决向量检索中的不同问题，而且经常可以组合使用。



| 方法 | 核心解决什么问题 |
|---|---|
| **Flat** | 最准确，但全量扫描慢 |
| **HNSW** | **怎么更快地找到可能相近的向量** |
| **IVF** | **怎么先缩小搜索范围，只搜几个区域** |
| **PQ** | **怎么把向量压缩，省内存、加速距离计算** |


```html
                 向量检索
                    │
       ┌────────────┼────────────┐
       │            │            │
    搜得快        少搜索       少占内存
       │            │            │
     HNSW          IVF           PQ
       
Flat = 什么技巧都不用，直接全部比较

```

> Flat 最大的好处：准。similar(chunk_vector_n, query_embedding), 每个 query 都扫描全部向量，会越来越贵。<br><br>
Exact Nearest Neighbor<br>
Faiss 官方也明确把 IndexFlatL2 作为精确搜索基线，不需要训练，也没有 ANN 参数。

> 不看所有向量，就大概率找到最近的？<br><br>
HNSW：不要遍历所有人，而是顺着“邻居关系”找<br>

Hierarchical Navigable Small World  分层可导航小世界图<br>
把所有向量组织成一个图，每个向量只连接一些附近的向量。搜索时顺着图不断向更相似的方向移动。

graph-based ANN

利用邻接图导航，而不是全量扫描。 -->  单层普通近邻图本身就很大呢？  -->  分层  多层图  -->
高层负责“远距离跳跃” ，底层负责“局部精细搜索” --> Hierarchical


Small World： 一个节点主要连接附近节点，但也保留一些能进行较远距离跳跃的连接。

> HNSW特点：既具有局部邻接，又具有长距离导航能力的图。
> 有一点点像`mini-batch 随机梯度下降`的思想。HNSW并不只是维护当前一个位置节点。

- 关键参数`efSearch`维护`candidate list`当前认为有希望的候选节点，维护搜索时愿意维护/探索多少候选。
- 关键参数`M`,每个节点邻居连接数量相关的参数 M, 图有多“密”。M大，连接更多，图更丰富、更容易找到正确路径、Recall 更高，代价就是内存和建索引成本增加。
- 关键参数`efConstruction`,建图时有多“认真”。efConstruction 小、随便找几个看起来不错的邻居 连起来 、 建得快、图质量可能稍差。
- 关键参数`efSearch`，查询时有多“认真”。

```
优点:
✓ 查询快
✓ Recall 高
✓ 不要求像 IVF 那样先训练聚类中心
✓ 参数相对直观
✓ 动态插入比较自然

代价：
✗ 图结构额外占内存
✗ 建索引有成本
✗ 数据特别巨大时内存压力明显
```

> IVF   Inverted File Index     倒排文件索引

先把整个向量空间分区   聚类、分区、找中心cluster-centroid  -> 根据query在对应的区里找，distance(query_embedding, cluster-centroid)  -> search in the cluster

- 重要参数`nlist`, 分多少个 cluster / bucket。
- 重要参数`nprobe`, 搜多少个 cluster / bucket。

> PQ    Product Quantization   乘积量化

一个 embedding 太大了，把长向量切成很多段。然后每个子空间做聚类。不再保存完整 float vector，而保存“每段最像哪个 centroid”。

> query 不一定也要真的解压所有向量。

> PQ 和 IVF 可以组合

```html
                    100M vectors
                         │
                         ↓
                    IVF 分区
                         │
            ┌────────────┼─────────────┐
            │            │             │
          List1        List2         List...
            │
            │
            └─────── 每个 vector
                        ↓
                     PQ压缩
```



> 这四个东西其实处在不同层

```html           
                    Vector Search
                          │
             ┌────────────┴────────────┐
             │                         │
        Search Strategy           Vector Storage
        搜哪些向量？               向量怎么存？
             │                         │
       ┌─────┼──────┐                  │
       │     │      │                  │
      Flat  HNSW   IVF                 PQ
       │     │      │                  │
    全部搜  图搜索  分区搜索             压缩
```



```html
Embedding Model
        ↓
      vectors
        ↓
┌──────────────────────┐
│    Vector Store      │
│                      │
│   Flat / HNSW / IVF  │
│         +            │
│       PQ/SQ          │
└──────────────────────┘
        ↓
    Top-K IDs
        ↓
    Documents

```

| | Flat | HNSW | IVF | PQ |
|---|---|---|---|---|
| 核心思想 | 全量扫描 | 图导航 | 聚类分桶 | 向量量化压缩 |
| 主要解决 | 精确基线 | 查询速度 | 减少候选数量 | 内存/计算 |
| Exact | ✅ | ❌ | ❌ | ❌ |
| 需要训练 | ❌ | ❌ | ✅ | ✅ |
| 额外内存 | 低 | **高** | 较低 | **大幅降低向量存储** |
| 典型参数 | 无 | `M`, `efSearch`, `efConstruction` | `nlist`, `nprobe` | `m`, `nbits` |
| 小中型 RAG | ✅ | **非常适合** | 通常没必要 | 通常没必要 |
| 超大规模 | 慢 | 内存可能高 | **适合** | **适合** |